# Week 3 Lab — Domain adaptation (walkthrough)

Three deliverables, three files under `src/adapt/`. Setup: `pip install -e ../common`,
`pip install -r requirements.txt`.

In [1]:
import sys; sys.path.insert(0, "src")
from adapt.schemas import Requirement, Missing
from adapt.router import CapabilityRouter
req = Requirement(description="handbook assistant, house style, 500k calls/mo, labelled data",
                  missing=Missing.behaviour, knowledge_changes=False,
                  knowledge_volume_tokens=1500, calls_per_month=500_000,
                  have_labeled_data=True, latency_sensitive=True)
try:
    print(CapabilityRouter().recommend(req))
except NotImplementedError as e:
    print("TODO:", e)
# then: !pytest -q tests/test_router.py

TODO: apply the decision rules (Day 07), in order:


## LoRA — prove the 'fine-tune' recommendation

Implement `LoRAAdapter` + `LoRATrainer` in `src/adapt/lora.py`. The test trains a rank-2 target and checks `r>=2` recovers it while `r=1` can't.

In [2]:
import numpy as np
from adapt.lora import LoRAAdapter, LoRATrainer
W = np.random.default_rng(0).standard_normal((16, 12)) / np.sqrt(12)
try:
    a = LoRAAdapter(W, r=2)
    print("n_trainable:", a.n_trainable, "| delta all-zero at init:", np.allclose(a.delta(), 0))
except NotImplementedError as e:
    print("TODO:", e)
# then: !pytest -q tests/test_lora.py

TODO: the low-rank weight update  (alpha/r) * B @ A   -> shape == W.shape


## Data audit + the live baseline

Implement `DatasetAudit`. Then with a key + `LLM_LIVE=1`, `test_live.py` runs one real few-shot Claude call and checks the router agrees fine-tuning would be cheaper at 500k calls/month.

In [3]:
# !pytest -q tests/test_audit.py
# !LLM_LIVE=1 pytest -q -m live -s